# PyTorch Zero to Hero

A detailed, ground-up PyTorch refresher: tensors through a full training loop and checkpointing —
with an encoder shaped like RoadFM-Lite's real one as the running example. Companion notebooks:
`refresher_numpy.ipynb`, `refresher_pandas.ipynb`, and `refresher_deep_learning.ipynb` (concepts:
backprop, attention, evaluation — read that one first if any term below is unfamiliar).

**PyTorch is not installed in this project's `.venv`** — CLAUDE.md rule 4 documents this as a known
gap: the model-training environment (used for `models/transformer_model.ipynb`) is separate and
currently undocumented. Every code cell below checks availability first and prints a skip message
instead of raising `ModuleNotFoundError`, so "Run All" finishes cleanly in this venv. Copy cells
into the training environment (or `pip install torch` into a dedicated venv) to actually run them.

**Contents**
1. Why PyTorch (vs. plain NumPy)
2. Tensors: creation, dtype, device
3. Tensor operations: elementwise, broadcasting, matmul, reshaping
4. In-place operations and memory
5. Autograd, in depth
6. Devices: CPU / CUDA / MPS
7. `nn.Module` — the building block
8. Common layers
9. Loss functions
10. Optimizers
11. `Dataset` and `DataLoader`
12. A full training loop, with validation
13. Checkpointing: save and load
14. Worked example: RoadFM-Lite's real architecture, MTR + TCP
15. Common pitfalls, summarized


In [ ]:
try:
    import torch
    import torch.nn as nn
    TORCH_AVAILABLE = True
    print(f"torch {torch.__version__} available")
except ImportError:
    torch = None
    nn = None
    TORCH_AVAILABLE = False
    print("torch not installed in this venv -- cells below print a note and skip instead of erroring.")


## 1. Why PyTorch

NumPy arrays (`refresher_numpy.ipynb`) are fast, vectorized, and CPU-only, with no memory of how
they were computed. A `torch.Tensor` adds exactly two things on top of that same array idea:

1. **A device** — the same tensor API runs on CPU, an NVIDIA GPU (`cuda`), or Apple Silicon
   (`mps`), by changing one `.to(device)` call.
2. **Autograd** — every operation on a tensor with `requires_grad=True` is recorded into a graph,
   so `.backward()` can compute *every* parameter's gradient automatically (§5). This is the
   "autograd framework" `refresher_deep_learning.ipynb` §11 describes in the abstract, in concrete
   PyTorch form.


## 2. Tensors: creation, dtype, device

Tensor construction mirrors NumPy almost one-for-one — if you know §2 of `refresher_numpy.ipynb`,
you already know most of this.


In [ ]:
if TORCH_AVAILABLE:
    import numpy as np

    a = torch.tensor([1.0, 2.0, 3.0])
    zeros = torch.zeros(2, 3)
    ones = torch.ones(2, 3)
    full = torch.full((2, 3), 7.0)
    arange = torch.arange(0, 10, 2)
    linspace = torch.linspace(0, 1, 5)
    randn = torch.randn(2, 3)          # standard normal, like np.random.default_rng().normal()

    print("a:", a, a.dtype)
    print("zeros:\n", zeros)
    print("randn:\n", randn)

    # NumPy <-> torch conversion is (usually) zero-copy on CPU -- they SHARE memory.
    np_arr = np.array([1.0, 2.0, 3.0], dtype=np.float32)
    from_np = torch.from_numpy(np_arr)
    from_np[0] = 999.0
    print("\nmutating the tensor also mutated the numpy array (shared memory):", np_arr)

    back_to_np = from_np.numpy()       # the reverse direction, also zero-copy on CPU
    print("round-trip is the same object family:", type(back_to_np).__name__)

    # dtype: torch defaults to float32 (NOT float64 like NumPy) -- this matters for matching
    # this project's X_windows.npy, which is also float32.
    print("\ndefault torch.tensor([1.0]).dtype:", torch.tensor([1.0]).dtype)
    f64 = torch.tensor([1.0], dtype=torch.float64)
    print("explicit float64:", f64.dtype)
    print("cast with .float()/.double()/.to(dtype):", f64.float().dtype)
else:
    print("skipped -- torch not installed")


## 3. Tensor operations: elementwise, broadcasting, matmul, reshaping

Same rules as NumPy §8–§12: elementwise ops broadcast; `@`/`matmul` is matrix multiplication;
`.reshape()`/`.view()` change shape without copying data when possible.


In [ ]:
if TORCH_AVAILABLE:
    x = torch.arange(12.0).reshape(3, 4)
    print("x:\n", x)
    print("x + 10 (broadcast scalar):\n", x + 10)

    row_vec = torch.tensor([1.0, 2.0, 3.0, 4.0])
    print("x + row_vec (broadcast (4,) against (3,4)):\n", x + row_vec)

    A = torch.randn(3, 4)
    B = torch.randn(4, 5)
    print("\nA @ B shape (matmul):", (A @ B).shape)
    print("A.T shape (transpose, a view):", A.T.shape)

    # .view() requires contiguous memory and shares storage; .reshape() works either way
    # (copies if it must). Prefer .reshape() unless you specifically need the "must be a view,
    # error if not" guarantee.
    flat = x.reshape(-1)
    print("\nflattened:", flat.shape)
    windowed = torch.arange(4 * 20 * 13).reshape(4, 20, 13)   # same shape convention as X_windows
    print("windowed shape:", windowed.shape)
    print("windowed.reshape(-1, 13) shape (collapse outer dims):", windowed.reshape(-1, 13).shape)

    # unsqueeze/squeeze -- add/remove size-1 axes, the torch names for NumPy's newaxis/squeeze.
    v = torch.tensor([1.0, 2.0, 3.0])
    print("\nv.unsqueeze(0) shape:", v.unsqueeze(0).shape)    # (1, 3)
    print("v.unsqueeze(1) shape:", v.unsqueeze(1).shape)       # (3, 1)

    # cat joins along an existing axis; stack creates a new one -- same distinction as NumPy §11.
    print("\ntorch.cat([v, v]) shape:", torch.cat([v, v]).shape)     # (6,)
    print("torch.stack([v, v]) shape:", torch.stack([v, v]).shape)    # (2, 3)
else:
    print("skipped -- torch not installed")


## 4. In-place operations and memory

Methods ending in `_` (e.g. `add_`, `mul_`, `zero_`) mutate the tensor **in place** instead of
returning a new one — saves memory, but is a common source of subtle autograd bugs: mutating a
tensor that autograd needs for the backward pass raises a runtime error, or worse, silently
corrupts a gradient computation if the check doesn't trigger.


In [ ]:
if TORCH_AVAILABLE:
    a = torch.tensor([1.0, 2.0, 3.0])
    b = a.add(1.0)     # NOT in place -- a is unchanged, b is new
    print("a after non-in-place add:", a)
    print("b (the new tensor):", b)

    a.add_(1.0)         # in place -- mutates a directly, returns a itself
    print("a after in-place add_:", a)

    # The classic autograd trap: mutating a leaf tensor that requires_grad, in place, after it's
    # already been used in the graph.
    x = torch.tensor([2.0], requires_grad=True)
    y = x * 3
    try:
        x.add_(1.0)      # torch detects this could corrupt the gradient computation for y
        y.backward()
    except RuntimeError as e:
        print("\nin-place mutation on a tensor already used in the graph raised:")
        print(str(e)[:150], "...")
else:
    print("skipped -- torch not installed")


## 5. Autograd, in depth

`requires_grad=True` tells torch to track a tensor's history. `.backward()` walks that history
*backwards* (reverse-mode autodiff, exactly `refresher_deep_learning.ipynb` §10's hand-written
backprop) and accumulates `d(loss)/d(param)` into each leaf tensor's `.grad`.


In [ ]:
if TORCH_AVAILABLE:
    x = torch.tensor([2.0, 3.0], requires_grad=True)
    y = (x ** 2).sum()          # y = x0^2 + x1^2
    y.backward()
    print("dy/dx (should be 2x):", x.grad)

    # Gradients ACCUMULATE across .backward() calls -- you must zero them yourself between steps.
    # This is exactly what optimizer.zero_grad() does in the training loop (§12).
    y2 = (x ** 2).sum()
    y2.backward()
    print("grad after a SECOND backward() without zeroing (doubled, not overwritten):", x.grad)
    x.grad.zero_()
    print("grad after manual zero_():", x.grad)

    # .detach() cuts a tensor off from the graph -- same values, but no gradient history. Use it
    # when you need a value (e.g. for logging or a numpy conversion) without keeping the graph alive.
    detached = (x ** 2).detach()
    print("\ndetached.requires_grad:", detached.requires_grad)

    # torch.no_grad(): a context where NO operations build a graph at all -- use during evaluation/
    # inference, where you'll never call .backward(). Saves memory and time.
    with torch.no_grad():
        no_grad_result = x * 2
    print("inside no_grad(), requires_grad:", no_grad_result.requires_grad)

    # Only leaf tensors (created directly, not computed) with requires_grad=True accumulate .grad
    # by default -- intermediate tensors' .grad is None unless you call .retain_grad() on them.
    a = torch.tensor([1.0], requires_grad=True)
    b = a * 2          # b is NOT a leaf -- it's computed from a
    c = (b ** 2).sum()
    c.backward()
    print("\nleaf tensor a.grad:", a.grad)
    print("non-leaf tensor b.grad (None unless .retain_grad() was called):", b.grad)
else:
    print("skipped -- torch not installed")


## 6. Devices: CPU / CUDA / MPS

The same code runs anywhere by parameterizing the device and calling `.to(device)` on **both** the
model and every batch of data — forgetting one of the two is the most common device-mismatch bug
("Expected all tensors to be on the same device").


In [ ]:
if TORCH_AVAILABLE:
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif torch.backends.mps.is_available():   # Apple Silicon
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
    print("selected device:", device)

    x = torch.randn(3, 3).to(device)          # move data...
    linear = nn.Linear(3, 3).to(device)       # ...and the model, to the SAME device
    print("output device matches input:", linear(x).device == x.device)

    # A device mismatch (model on one device, data on another) raises immediately -- fails loud,
    # not silently wrong -- e.g. on a machine with a GPU:
    #     model = nn.Linear(3, 3).to("cuda")
    #     model(torch.randn(3, 3))              # data still on CPU -> RuntimeError
    # On this machine only one device is available, so there is nothing to mismatch against; the
    # habit that avoids the bug is calling .to(device) on BOTH the model and every batch, always.
else:
    print("skipped -- torch not installed")


## 7. `nn.Module` — the building block

Every layer, and the whole model, subclasses `nn.Module` and implements `forward()`. `nn.Module`
automatically tracks every parameter and sub-module you assign as an attribute — that's what makes
`model.parameters()` (§10) and `model.to(device)` (§6) recurse through the whole thing for free.


In [ ]:
if TORCH_AVAILABLE:
    class TwoLayerNet(nn.Module):
        def __init__(self, in_dim=13, hidden=32, out_dim=1):
            super().__init__()                       # ALWAYS call this first
            self.fc1 = nn.Linear(in_dim, hidden)
            self.act = nn.ReLU()
            self.fc2 = nn.Linear(hidden, out_dim)

        def forward(self, x):
            return self.fc2(self.act(self.fc1(x)))

    net = TwoLayerNet()
    print(net)                                         # nn.Module has a nice repr for free
    print("\nnamed parameters:")
    for name, p in net.named_parameters():
        print(f"  {name}: shape {tuple(p.shape)}, requires_grad={p.requires_grad}")

    x = torch.randn(5, 13)      # batch of 5
    out = net(x)                 # calling net(x) invokes forward() -- never call .forward() directly
    print("\noutput shape:", out.shape)
    print("total trainable params:", sum(p.numel() for p in net.parameters()))

    # .train() / .eval() toggle layers that behave differently at train vs. inference time
    # (Dropout, BatchNorm). This tiny net has neither, but ALWAYS set the mode explicitly in real
    # code -- forgetting .eval() before validation is a classic, hard-to-notice bug.
    net.train()
    print("\nnet.training after .train():", net.training)
    net.eval()
    print("net.training after .eval():", net.training)
else:
    print("skipped -- torch not installed")


## 8. Common layers

The handful of `nn` building blocks that cover most architectures, including RoadFM-Lite's encoder.


In [ ]:
if TORCH_AVAILABLE:
    linear = nn.Linear(13, 128)                 # y = xW^T + b -- the input projection RoadFM-Lite uses
    ln = nn.LayerNorm(128)                        # normalizes across the FEATURE dim, per-sample
    dropout = nn.Dropout(p=0.1)                    # zeroes ~10% of activations at train time only
    embedding = nn.Embedding(num_embeddings=1000, embedding_dim=64)   # integer id -> dense vector

    x = torch.randn(5, 20, 13)                    # (batch, timesteps, features) -- X_windows' shape
    projected = linear(x)                          # Linear applies to the LAST dim only: (5, 20, 128)
    print("linear output shape:", projected.shape)
    print("layernorm output shape:", ln(projected).shape)

    dropout.train()
    print("dropout zeroes some entries at train time:", (dropout(projected) == 0).float().mean().item() > 0)
    dropout.eval()
    print("dropout is a no-op in eval mode:", torch.equal(dropout(projected), projected))

    # MultiheadAttention -- the core operation inside a transformer encoder layer.
    mha = nn.MultiheadAttention(embed_dim=128, num_heads=8, batch_first=True)
    attn_out, attn_weights = mha(projected, projected, projected)   # self-attention: Q=K=V
    print("\nattention output shape:", attn_out.shape)
    print("attention weights shape (batch, tgt_len, src_len):", attn_weights.shape)

    # TransformerEncoderLayer bundles self-attention + feedforward + residuals + layernorm into one
    # block (refresher_deep_learning.ipynb §15); TransformerEncoder stacks N of them.
    layer = nn.TransformerEncoderLayer(d_model=128, nhead=8, dim_feedforward=512, batch_first=True)
    encoder = nn.TransformerEncoder(layer, num_layers=6)   # RoadFM-Lite's real depth, per plan §3.1
    encoded = encoder(projected)
    print("\n6-layer TransformerEncoder output shape:", encoded.shape)   # same shape as input
else:
    print("skipped -- torch not installed")


## 9. Loss functions

Which loss to use depends on the task — exactly `refresher_deep_learning.ipynb` §3's table, now as
PyTorch calls.


In [ ]:
if TORCH_AVAILABLE:
    # MSELoss -- regression / reconstruction. RoadFM-Lite's MTR head uses this.
    mse = nn.MSELoss()
    pred = torch.randn(5, 13)
    target = torch.randn(5, 13)
    print("MSE loss:", mse(pred, target).item())

    # CrossEntropyLoss -- multi-class classification. Takes RAW LOGITS (not softmax probabilities!)
    # and INTEGER class labels (not one-hot) -- both common mistakes.
    ce = nn.CrossEntropyLoss()
    logits = torch.randn(5, 5)              # (batch=5, num_classes=5) -- RoadFM-Lite's 5-class head
    labels = torch.tensor([0, 4, 2, 2, 1])  # integer class index per sample, not one-hot
    print("cross-entropy loss:", ce(logits, labels).item())

    # BCEWithLogitsLoss -- binary classification, also takes raw logits (combines sigmoid + BCE
    # internally, which is more numerically stable than doing them as two separate steps).
    bce = nn.BCEWithLogitsLoss()
    binary_logits = torch.randn(5)
    binary_labels = torch.tensor([0.0, 1.0, 1.0, 0.0, 1.0])
    print("BCE-with-logits loss:", bce(binary_logits, binary_labels).item())

    # A loss is a SCALAR tensor with its own grad_fn -- that's what .backward() is called on.
    print("\nMSE loss requires_grad:", mse(torch.randn(3, requires_grad=True), torch.randn(3)).requires_grad)
else:
    print("skipped -- torch not installed")


## 10. Optimizers

An optimizer holds a reference to a set of parameters and updates them from their `.grad` after
`.backward()`. `Adam`/`AdamW` are the default choice for transformers; plain `SGD` is the simplest
and most literal implementation of `refresher_deep_learning.ipynb` §1's `θ ← θ − η∇L`.


In [ ]:
if TORCH_AVAILABLE:
    import torch.optim as optim

    net = nn.Linear(4, 1)
    sgd = optim.SGD(net.parameters(), lr=0.01)
    adam = optim.Adam(net.parameters(), lr=1e-3)                  # adapts the LR per parameter
    adamw = optim.AdamW(net.parameters(), lr=1e-3, weight_decay=0.01)  # Adam + decoupled L2 (§5 of
                                                                          # refresher_deep_learning.ipynb)

    x = torch.randn(8, 4)
    y = torch.randn(8, 1)
    loss_fn = nn.MSELoss()

    # One full optimizer step, spelled out -- this triplet is the heart of every training loop (§12).
    adamw.zero_grad()                 # 1. clear old gradients (they ACCUMULATE otherwise, see §5)
    loss = loss_fn(net(x), y)
    loss.backward()                   # 2. compute new gradients via autograd
    adamw.step()                      # 3. update parameters using those gradients
    print("loss after one step:", loss.item())

    # A learning-rate scheduler adjusts lr over training -- e.g. warmup then decay, common for
    # transformers. StepLR is the simplest: multiply lr by gamma every step_size epochs.
    scheduler = optim.lr_scheduler.StepLR(adamw, step_size=10, gamma=0.5)
    print("initial lr:", scheduler.get_last_lr())
    for _ in range(10):
        scheduler.step()
    print("lr after 10 scheduler steps:", scheduler.get_last_lr())
else:
    print("skipped -- torch not installed")


## 11. `Dataset` and `DataLoader`

`Dataset` wraps a source of examples (`__len__` + `__getitem__`); `DataLoader` batches, shuffles,
and (optionally) parallelizes loading across worker processes. This is the shape `X_windows.npy` +
`idx_train.npy` get consumed in.


In [ ]:
if TORCH_AVAILABLE:
    import numpy as np
    from torch.utils.data import Dataset, DataLoader

    class WindowDataset(Dataset):
        """Label-free: __getitem__ returns a window only, no label -- per the label-free
        pretraining constraint (memory: project_label_free_pretraining). Labels are added only in a
        separate Dataset used for fine-tuning heads."""

        def __init__(self, windows: np.ndarray):
            self.windows = torch.from_numpy(windows).float()

        def __len__(self):
            return len(self.windows)

        def __getitem__(self, idx):
            return self.windows[idx]

    toy_windows = np.random.default_rng(0).normal(size=(40, 20, 13)).astype(np.float32)
    dataset = WindowDataset(toy_windows)
    print("dataset length:", len(dataset))
    print("dataset[0] shape:", dataset[0].shape)

    loader = DataLoader(dataset, batch_size=8, shuffle=True, drop_last=False)
    print("\nnumber of batches:", len(loader))
    for i, batch in enumerate(loader):
        print(f"batch {i}: shape {tuple(batch.shape)}")
        if i == 1:
            break

    # A labeled Dataset for fine-tuning: __getitem__ returns (window, label) instead.
    class LabeledWindowDataset(Dataset):
        def __init__(self, windows: np.ndarray, labels: np.ndarray):
            self.windows = torch.from_numpy(windows).float()
            self.labels = torch.from_numpy(labels).long()

        def __len__(self):
            return len(self.windows)

        def __getitem__(self, idx):
            return self.windows[idx], self.labels[idx]

    toy_labels = np.random.default_rng(0).integers(0, 5, size=40)
    labeled = LabeledWindowDataset(toy_windows, toy_labels)
    labeled_loader = DataLoader(labeled, batch_size=8, shuffle=True)
    xb, yb = next(iter(labeled_loader))
    print("\nlabeled batch shapes:", tuple(xb.shape), tuple(yb.shape))
else:
    print("skipped -- torch not installed")


## 12. A full training loop, with validation

Every piece from §5–§11, assembled: train on batches with gradients, evaluate on a held-out split
with `torch.no_grad()` and `.eval()` mode, track both losses.


In [ ]:
if TORCH_AVAILABLE:
    import numpy as np
    from torch.utils.data import DataLoader

    torch.manual_seed(0)
    n = 200
    X = torch.randn(n, 13)
    y = (X.sum(dim=1) > 0).long()                  # a toy binary-ish target derived from X

    n_train = 160
    train_ds = torch.utils.data.TensorDataset(X[:n_train], y[:n_train])
    val_ds = torch.utils.data.TensorDataset(X[n_train:], y[n_train:])
    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)

    model = nn.Sequential(nn.Linear(13, 32), nn.ReLU(), nn.Linear(32, 2))
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(5):
        model.train()                                  # enable train-mode layers (none here, but always do this)
        train_loss = 0.0
        for xb, yb in train_loader:
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * len(xb)
        train_loss /= len(train_ds)

        model.eval()                                    # disable train-mode layers for evaluation
        val_loss = 0.0
        correct = 0
        with torch.no_grad():                            # no graph needed -- saves memory and time
            for xb, yb in val_loader:
                logits = model(xb)
                val_loss += loss_fn(logits, yb).item() * len(xb)
                correct += (logits.argmax(dim=1) == yb).sum().item()
        val_loss /= len(val_ds)
        val_acc = correct / len(val_ds)

        print(f"epoch {epoch}: train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  val_acc={val_acc:.2f}")
else:
    print("skipped -- torch not installed")


## 13. Checkpointing: save and load

Save `.state_dict()` — a plain dict of tensor names to values — not the whole model object; it's
portable across code changes as long as the architecture matches when you reload. This is exactly
the `roadfm_lite_pretrained.pt` / `roadfm_lite_final.pt` pattern (CLAUDE.md's "Checkpoints" section).


In [ ]:
if TORCH_AVAILABLE:
    import tempfile, pathlib

    model = nn.Linear(13, 5)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    with tempfile.TemporaryDirectory() as tmp:
        ckpt_path = pathlib.Path(tmp) / "checkpoint.pt"

        # Save model + optimizer state + arbitrary metadata together -- reproducibility (rule 4).
        torch.save({
            "epoch": 3,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "config": {"input_dim": 13, "d_model": 128, "seed": 42},
        }, ckpt_path)
        print("checkpoint saved, size:", ckpt_path.stat().st_size, "bytes")

        # Reloading requires an already-constructed model with the SAME architecture --
        # state_dict only holds weights, not the architecture definition itself.
        fresh_model = nn.Linear(13, 5)
        checkpoint = torch.load(ckpt_path, weights_only=False)
        fresh_model.load_state_dict(checkpoint["model_state_dict"])
        print("reloaded epoch:", checkpoint["epoch"])
        print("reloaded config:", checkpoint["config"])
        print("weights match after reload:", torch.equal(
            dict(model.named_parameters())["weight"],
            dict(fresh_model.named_parameters())["weight"],
        ))

        # model.eval() after loading for inference -- easy to forget, changes Dropout/BatchNorm behavior.
        fresh_model.eval()
else:
    print("skipped -- torch not installed")


## 14. Worked example: RoadFM-Lite's real architecture, MTR + TCP

Assembling §7–§12 into the project's actual encoder config — `input_dim=13, d_model=128, nhead=8,
num_layers=6, dim_feedforward=512, dropout=0.1` (CLAUDE.md §3.1) — with both pretraining
objectives: **MTR** (masked reconstruction) and **TCP** (clean-vs-corrupted classification).


In [ ]:
if TORCH_AVAILABLE:
    import math

    class SinusoidalPE(nn.Module):
        """Fixed (non-learned) positional encoding -- fine since windows are always exactly 20
        steps, never extrapolated to a longer sequence at inference time."""

        def __init__(self, d_model, max_len=20):
            super().__init__()
            pos = torch.arange(max_len).unsqueeze(1).float()
            div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
            pe = torch.zeros(max_len, d_model)
            pe[:, 0::2] = torch.sin(pos * div)
            pe[:, 1::2] = torch.cos(pos * div)
            self.register_buffer("pe", pe)   # not a parameter -- not trained, but moves with .to(device)

        def forward(self, x):                 # x: (batch, seq, d_model)
            return x + self.pe.unsqueeze(0)

    class RoadFMLiteEncoder(nn.Module):
        """The encoder half of RoadFM-Lite, per docs/plan/research-plan.md §3.1."""

        def __init__(self, input_dim=13, d_model=128, nhead=8, num_layers=6, dim_feedforward=512, dropout=0.1):
            super().__init__()
            self.input_proj = nn.Linear(input_dim, d_model)
            self.pos_encoding = SinusoidalPE(d_model)
            layer = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout, batch_first=True)
            self.encoder = nn.TransformerEncoder(layer, num_layers)

        def forward(self, x):                  # x: (batch, 20, 13)
            h = self.pos_encoding(self.input_proj(x))
            return self.encoder(h)               # (batch, 20, d_model) -- per-step representations

    encoder = RoadFMLiteEncoder()
    n_params = sum(p.numel() for p in encoder.parameters())
    print(f"encoder parameters: {n_params:,}")     # report this number in the thesis (rule 4)

    # MTR head: reconstruct the raw 13 features at each (masked) timestep.
    mtr_head = nn.Linear(128, 13)
    # TCP head: classify a whole window as clean vs. one of the corruption types.
    tcp_head = nn.Linear(128, 5)   # identity, replay, shuffle, speed-scale, position-offset

    batch = torch.randn(4, 20, 13)

    # --- MTR: mask 15% of timesteps, reconstruct, MSE on masked positions only ---
    mask = torch.rand(4, 20) < 0.15
    masked_batch = batch.clone()
    masked_batch[mask] = 0.0                        # a real model learns a mask TOKEN instead of zeros
    h_mtr = encoder(masked_batch)
    recon = mtr_head(h_mtr)
    mtr_loss = ((recon - batch) ** 2)[mask].mean()
    print("MTR loss:", mtr_loss.item())

    # --- TCP: classify pooled window embeddings ---
    h_tcp = encoder(batch)
    pooled = h_tcp.mean(dim=1)                        # mean-pool over the 20 steps -> (4, 128)
    tcp_logits = tcp_head(pooled)
    tcp_labels = torch.zeros(4, dtype=torch.long)      # toy: pretend every window is "identity" (clean)
    tcp_loss = nn.functional.cross_entropy(tcp_logits, tcp_labels)
    print("TCP loss:", tcp_loss.item())

    # --- Combined pretraining objective, per docs/plan/research-plan.md: L = λ1*L_MTR + λ2*L_TCP ---
    lambda_mtr, lambda_tcp = 1.0, 0.5
    total_loss = lambda_mtr * mtr_loss + lambda_tcp * tcp_loss
    print("combined pretraining loss:", total_loss.item())

    optimizer = torch.optim.AdamW(
        list(encoder.parameters()) + list(mtr_head.parameters()) + list(tcp_head.parameters()), lr=1e-4)
    optimizer.zero_grad()
    total_loss.backward()
    optimizer.step()
    print("\none combined MTR+TCP optimizer step completed")
else:
    print("skipped -- torch not installed")


## 15. Common pitfalls, summarized

- **Forgetting `optimizer.zero_grad()`** (§5, §10) — gradients accumulate across `.backward()`
  calls by default; without zeroing, each step's update is contaminated by every previous step's
  gradient too.
- **Forgetting `.eval()` before validation / inference** (§7, §12) — leaves Dropout/BatchNorm in
  training mode, so validation numbers become noisy and non-reproducible run to run.
- **Forgetting `torch.no_grad()` during evaluation** (§5, §12) — still *works*, but wastes memory
  and time building a graph you'll never call `.backward()` on.
- **Passing softmax probabilities to `CrossEntropyLoss`** (§9) — it expects raw logits and applies
  log-softmax internally; feeding it already-softmaxed values silently gives a wrong loss.
- **Device mismatches** (§6) — model and data must be on the *same* device; `.to(device)` both,
  every time, including inside custom `Dataset.__getitem__` if you preload tensors.
- **In-place ops on tensors autograd still needs** (§4) — usually raises a clear `RuntimeError`,
  but treat it as a sign to use the non-`_` variant instead of working around the error.
- **`.item()` vs. keeping a tensor** — call `.item()` (or `.detach()`) before accumulating a running
  loss in a Python float/list; keeping the tensor itself keeps its whole graph alive, growing
  memory every iteration ("gradient graph leak").
- **`torch.load` and arbitrary code execution** — a `.pt` file can contain pickled Python objects;
  only load checkpoints from sources you trust, and prefer `weights_only=True` when you only need
  tensors (PyTorch ≥ 2.4 defaults to this).
